# GDEX Services

```{image} ../thumbnails/gdex_logo.png
:alt: NSF NCAR GDEX logo
:width: 300px
```



---

## Introduction

The [GDEX (Geoscience Data Exchange)](https://gdex.ucar.edu) comes packaged with value added services that can assist a researcher in a variety of ways.

This notebook gives an overview of the access services available through GDEX and when to use each one. For hands-on examples of working with specific datasets and more detailed examples, see the other notebooks in this cookbook.

## Services and Tools

GDEX offers a comprehensive suite of services to help researchers at every stage of their workflow.

### Discovery and Search

The [GDEX web portal search](https://gdex.ucar.edu/gsearch/dataset-search) provides tools for finding data:

- **Full-text and faceted search** across thousands of datasets with filtering by variable, domain, date range, and format
- **Rich metadata** including dataset documentation, version history, and related publications

### Data Access Pathways

GDEX meets researchers wherever they work:

| Pathway | Best For |
|---|---|
| **Web download** | Browsing and file search allow full downloads via the portal |
| **Globus transfer** | Bulk, reliable transfer of large datasets to HPC systems |
| **THREDDS OPeNDAP/NCSS** | Programmatic and subsetting access from scripts |
| **Cloud streaming (Zarr/Kerchunk)** | Analysis-ready access without downloading |
| **API** | Integration directly into data science workflows to bypass gdex web portal |


:::{tip}
For most workflows: use **web download** for exploration, **Globus** once you know which files you need and they exceed a few GB, and **OPeNDAP or Zarr** when you only need a spatial or temporal subset.
:::

### Subsetting and Format Conversion

Many datasets support server-side subsetting, allowing you to request only the variables, spatial domain, and time range you need. Have our HPC do the hard work and reduce download size and time significantly.

### Compute-Proximate Access

For large datasets where moving data is impractical, GDEX data is accessible from NSF NCAR computing resources including the **Derecho** supercomputer and the **Casper** analysis cluster. Researchers with allocations can run analyses directly alongside the data.


:::{note}
Compute-proximate access requires an active NSF NCAR HPC allocation. Researchers without allocations can still reach all datasets via the other pathways listed above.
:::

### THREDDS Data Server (TDS)

GDEX exposes many datasets through a [THREDDS Data Server (TDS)](https://www.unidata.ucar.edu/software/tds/), which provides OPeNDAP, HTTP, and WMS endpoints from a single catalog. TDS is a good choice when you want to:

- **Subset on the server side** — request only the variables, levels, times, and spatial domain you need, without downloading full files
- **Explore datasets programmatically** — browse catalogs in Python using [Siphon](https://unidata.github.io/siphon/) before committing to a download
- **Integrate with standard tools** — TDS endpoints are compatible with Panoply, NCO, CDO, Ferret, and any tool that speaks OPeNDAP


:::{seealso}
`tds_xarray.ipynb` and `tds_siphon.ipynb` in the Services section show hands-on TDS and Siphon examples.
:::

### API

The [GDEX REST API](https://api.gdex.ucar.edu/documentation/) provides programmatic access to dataset metadata, file listings, and data request workflows — letting you integrate GDEX into scripts and pipelines without using the web portal.

**Base URL:** `https://api.gdex.ucar.edu/`

#### What the API Can Do

| Capability | Endpoints |
|---|---|
| **Dataset metadata** | Abstract, variables, spatial/temporal coverage, formats, license, publications |
| **File listings** | Browse file groups and get lists of web-accessible files |
| **Data requests** | Submit subsetting jobs, poll status, retrieve output files |
| **ARCO data** | Check for Analysis-Ready Cloud-Optimized availability and browse variables |
| **Globus transfers** | Initiate a Globus transfer for a completed request |
| **Metrics** | Usage statistics, download volume, citation counts |

Most read endpoints are publicly accessible. Submitting data requests requires a free [RDA account](https://rda.ucar.edu/index.html#!cgi-bin/login).

#### Example: Fetching Dataset Metadata

```python
import requests

dsid = "d633000"  # ERA5
base = "https://api.gdex.ucar.edu"

# Get temporal and spatial coverage
temporal = requests.get(f"{base}/datasets/{dsid}/temporal/").json()
spatial  = requests.get(f"{base}/datasets/{dsid}/spatial_coverage/").json()
variables = requests.get(f"{base}/datasets/{dsid}/variables/").json()
```

#### Example: Submitting a Subsetting Request

Subsetting requests are submitted as JSON and processed asynchronously. Use `GET /status/{rindex}/` to poll for completion, then `GET /get_req_files/{rindex}/` to retrieve the output files.

```python
import requests, time, os

session = requests.Session()
session.auth = (os.environ["RDA_EMAIL"], os.environ["RDA_PASSWORD"])

payload = {
    "dsid": "d083002",
    "date": "202301010000/202301310000",
    "param": "TMP/HGT",
    "level": "ISBL:500/850",
    "oformat": "netCDF",
    "nlat": 60.0, "slat": 20.0,
    "elon": -60.0, "wlon": -140.0,
}
resp = session.post("https://api.gdex.ucar.edu/submit_json/", json=payload)
rindex = resp.json()["data"]["rindex"]

# Poll until complete
while True:
    status = session.get(f"https://api.gdex.ucar.edu/status/{rindex}/").json()
    if status["data"]["status"] in ("Completed", "Error"):
        break
    time.sleep(30)
```

Full endpoint reference: [api.gdex.ucar.edu/documentation](https://api.gdex.ucar.edu/documentation/)


:::{seealso}
`api_metadata.ipynb` and `api_subset.ipynb` in the Services section cover the full API workflow.
:::

### Metadata and Documentation

Each dataset includes:

- Dataset-level metadata including abstract, variables, resources, and citations
- File-level metadata following community standards and uses [CF Conventions](http://cfconventions.org/) when possible
- Documentation and software files when applicable
- Version tracking so analyses can be reproduced precisely
- Links to related publications and data sources
